In [ ]:
%pip install --quiet pymongo mongomock faker vaderSentiment pandas matplotlib ipytest

# NoSQL Databases & ETL Pipelines: Giving Unstructured Data a Home

In Topics 1–4 you learned to *collect* unstructured data — scraping pages, calling APIs, parsing JSON and XML. You ended each of those sessions with a pile of raw, messy records. This topic answers the obvious next question:

**Where does that data actually live, and how do you get it there reliably?**

Relational tables — rows and columns with a fixed schema — are a poor fit for the varied, nested, half-structured records that collection produces. In this notebook we'll use a **document database** (MongoDB) and build a proper **ETL pipeline** (Extract → Transform → Load) to turn heterogeneous support records into a clean, queryable store.

By the end you'll be able to:

- Explain when a NoSQL document store is a better fit than a relational database — **and when it isn't**
- Distinguish **ETL** from **ELT** and justify a choice
- Build a repeatable Extract → Transform → Load pipeline in Python
- Load documents into MongoDB, index them, and run **aggregation pipelines** for insight
- Make the pipeline **idempotent, tested and reproducible** — so it produces the same result every run

```text
  RAW SOURCES              EXTRACT        TRANSFORM              LOAD            ANALYSE
  chat  ┐                                clean · normalise
  email ┼──▶  collect  ──▶  cache  ──▶   enrich · validate  ──▶  MongoDB  ──▶  aggregate
  form  ┘                                                        (documents)    · visualise
```

> This notebook is fully scaffolded — **run each cell** and read the narrative between them. You're not expected to type the code from scratch; you're expected to understand *why* each step is there.

## Where this topic sits

| Topics 1–4 | **Topic 5 (here)** | Topic 6 onwards |
|---|---|---|
| *Collect* raw unstructured data (scraping, APIs, JSON/XML) | *Store & pipeline* it into an analysis-ready form | *Analyse* it (NLP, LLMs, audio/image) |

Everything downstream — text analytics, topic modelling, LLM enrichment — assumes the data is already sitting somewhere clean and consistent. That "somewhere" and the pipeline that fills it are the subject of this topic.

We'll follow a single organisation throughout: **Pinnacle Broadband**, a UK internet provider whose customer-support data we'll carry forward into later topics.

## 🏫 Why NoSQL for unstructured data?

A relational database enforces a **schema**: every row of a table has the same columns, of the same types. That's a strength for well-understood, uniform data (a payments ledger, a stock table) and a genuine liability for the kind of data collection produces:

- A live-chat transcript is a *list of turns*. An email is a subject and a body. A web form is a flat set of fields. These describe the same thing — a customer contact — but have **completely different shapes**.
- Fields appear and disappear. Some emails have a subject line; some don't. Some form submissions include a satisfaction rating; some are left blank.
- The structure **evolves**. Next quarter Pinnacle adds a WhatsApp channel with fields nobody has seen before.

Forcing this into rigid tables means either a forest of mostly-empty columns or a tangle of join tables. A **document store** takes a different bargain: each record is a flexible, self-describing document (think JSON), and documents in the same collection needn't have identical fields.

### The four NoSQL families

| Family | Stores data as | Good for | Example |
|---|---|---|---|
| **Document** | JSON-like documents | Varied, nested, evolving records | MongoDB |
| **Key–value** | Simple key → value pairs | Caching, sessions, fast lookups | Redis |
| **Column-family** | Rows with dynamic columns | Huge write-heavy workloads | Cassandra |
| **Graph** | Nodes and relationships | Highly connected data | Neo4j |

For collected unstructured data, the **document** family is almost always the natural home — so that's what we'll use.

> 📌 **Assessment connection**
>
> This topic is a strong fit for **LO2 (processing unstructured data)** and **LO5 (reproducible analysis and robust code)**, with **LO4 (Python for analysis and visualisation)** exercised in the aggregation and charting steps. The apprenticeship standard explicitly requires you to understand *"database systems including relational, data warehousing, NoSQL and real-time approaches; the pros and cons of each"* — the appraisal sections below are where you demonstrate exactly that.

## 🏫 ETL vs ELT — two shapes of the same idea

**ETL — Extract, Transform, Load.** Pull the raw data, clean and reshape it *in your pipeline*, then load the finished result into the store. The store only ever holds tidy data.

**ELT — Extract, Load, Transform.** Load the raw data *first*, then transform it inside the store (or a downstream warehouse). The store holds both raw and derived data.

Neither is "correct" — they trade different things:

| | ETL | ELT |
|---|---|---|
| Store holds | Only clean data | Raw **and** derived data |
| Transform runs | In the pipeline, before load | In the store, after load |
| Best when | Transforms are stable; store is expensive; you want a tidy store | Raw data is valuable to keep; transforms change often; store is cheap and powerful |
| Reproducibility | Re-run the pipeline | Re-run the in-store transform |

> 🔍 **Design decision — which one for Pinnacle?**
>
> We'll build an **ETL** pipeline here, for two reasons that are worth stating out loud (this is exactly the kind of reasoning the technical report rewards):
>
> 1. Our transforms — normalising three channel shapes into one, deriving sentiment — are **well understood and stable**, so doing them once in the pipeline is efficient.
> 2. We want the store to hold a single, clean, consistent document shape that the *analysis* topics can rely on without re-cleaning.
>
> The honest counter-argument: if Pinnacle later wanted to keep every raw record verbatim for audit, or expected the enrichment logic to change monthly, **ELT would be the better call** — load raw, transform on read. Recognising *when your own choice would be wrong* is the mark of a Level 6 answer.

## The scenario: Pinnacle Broadband's support data

Pinnacle Broadband receives customer contacts through **three channels**, each with its own system and its own data shape:

- **Live chat** — a session with a list of conversational turns and an ISO timestamp
- **Email** — a subject, a body, a UK-formatted date string, sometimes with fields missing
- **Web contact form** — a flat record with an issue type, a satisfaction score and a Unix-epoch timestamp

These three feeds are the *output* of the collection work you did in Topics 1–4. Our job is to pipeline them into one clean MongoDB collection. Let's begin.

## 🏫 Step 1 — Extract

The Extract stage *gathers* raw data from its sources. Here we **synthesise** the three feeds so the notebook is fully reproducible, using a fixed random seed and a curated pool of realistic support phrases. In your own work the sources would be live scrapers or API calls — but the principle is identical: get the raw records, then **cache them**, so downstream steps (and re-runs) don't depend on the source being available.

In [ ]:
import random, json, os
from datetime import datetime, timedelta
from typing import Any
from collections.abc import Callable
from faker import Faker

# --- Type aliases -----------------------------------------------------------
# Readable names for the loosely-typed, JSON-like dictionaries this pipeline
# passes around. `Record` = a raw incoming record (any channel shape);
# `Document` = a canonical, cleaned document ready for the store. Aliases like
# these keep the function signatures below short and self-explanatory.
Record = dict[str, Any]
Document = dict[str, Any]

# --- Reproducibility: fixed seeds so every run produces identical raw data ---
SEED = 42
random.seed(SEED)
Faker.seed(SEED)
fake = Faker("en_GB")           # UK names, emails, companies

CACHE_DIR = "pinnacle_raw"      # cache-first: raw feeds are saved here on first run
os.makedirs(CACHE_DIR, exist_ok=True)

# Curated content pools — template-based generation with a known 'tone' per phrase,
# so the derived sentiment later is meaningful rather than random.
ISSUE_TYPES = ["outage", "slow_speed", "billing", "installation", "cancellation", "router_fault"]

OPENERS = {
    "negative": ["Absolutely appalling.", "I am furious.", "This is completely unacceptable.",
                 "Really disappointed with Pinnacle.", "I have had enough."],
    "neutral":  ["Quick question.", "Just checking on something.", "Wondering if you can help.",
                 "A query about my account.", "Hoping you can clarify."],
    "positive": ["Brilliant service, thank you.", "Really happy with the help.",
                 "Fantastic support today.", "Thank you so much, sorted quickly.", "Delighted with Pinnacle."],
}

ISSUE_PHRASE = {
    "outage":       "My broadband has been completely down for hours.",
    "slow_speed":   "The connection is painfully slow and keeps dropping.",
    "billing":      "I have been charged twice on my latest bill.",
    "installation": "The engineer never turned up for the installation.",
    "cancellation": "I want to cancel my contract and leave.",
    "router_fault": "The router keeps restarting itself.",
}

def compose(tone: str, issue: str) -> str:
    """Compose a support message from a known tone and issue type.

    Building each message from a fixed tone lets us reason about the sentiment
    the Transform stage will later derive, keeping the notebook predictable.

    Args:
        tone: One of "negative", "neutral" or "positive".
        issue: One of the values in ``ISSUE_TYPES``.

    Returns:
        A single message string combining an opener and an issue phrase.
    """
    return f"{random.choice(OPENERS[tone])} {ISSUE_PHRASE[issue]}"

print("Extract configured — seed fixed, cache directory ready.")

In [ ]:
N_CHAT, N_EMAIL, N_WEBFORM = 60, 45, 55   # deliberately uneven volumes per channel
BASE = datetime(2026, 3, 1, 8, 0, 0)

def make_chat_feed(n: int) -> list[Record]:
    """Generate *n* synthetic live-chat sessions (shape #1: a list of turns).

    Args:
        n: Number of chat sessions to create.

    Returns:
        A list of raw chat records, each with a session id, ISO timestamp,
        customer reference and a short transcript.
    """
    feed: list[Record] = []
    for i in range(n):
        tone = random.choices(["negative", "neutral", "positive"], weights=[5, 3, 2])[0]
        issue = random.choice(ISSUE_TYPES)
        opening = compose(tone, issue)
        feed.append({
            "session_id": f"CH-{10000 + i}",
            "started": (BASE + timedelta(minutes=random.randint(0, 40000))).isoformat(),
            "customer_ref": f"PB-{random.randint(10000, 99999)}",
            "transcript": [
                {"role": "customer", "text": opening},
                {"role": "agent", "text": "Thanks for getting in touch, let me take a look."},
            ],
        })
    return feed

chat_feed = make_chat_feed(N_CHAT)
print(f"{len(chat_feed)} chat sessions")
print(json.dumps(chat_feed[0], indent=2))

In [ ]:
def make_email_feed(n: int) -> list[Record]:
    """Generate *n* synthetic emails (shape #2: subject/body, UK date string).

    Roughly one email in eight is created without a subject line, to give the
    quality gate something realistic to catch later.

    Args:
        n: Number of emails to create.

    Returns:
        A list of raw email records.
    """
    feed: list[Record] = []
    for i in range(n):
        tone = random.choices(["negative", "neutral", "positive"], weights=[5, 3, 2])[0]
        issue = random.choice(ISSUE_TYPES)
        dt = BASE + timedelta(minutes=random.randint(0, 40000))
        rec: Record = {
            "message_id": f"<{fake.uuid4()}@mail.pinnacle.co.uk>",
            "from": fake.email(),
            "subject": f"Re: {issue.replace('_', ' ')}",
            "body": compose(tone, issue),
            "date": dt.strftime("%d/%m/%Y %H:%M"),   # UK dd/mm/yyyy — different format!
            "account": f"PB-{random.randint(10000, 99999)}",
        }
        # ~1 in 8 emails arrives with no subject line — a data-quality reality
        if random.random() < 0.12:
            del rec["subject"]
        feed.append(rec)
    return feed

email_feed = make_email_feed(N_EMAIL)
print(f"{len(email_feed)} emails")
print(json.dumps(email_feed[0], indent=2))

In [ ]:
def make_webform_feed(n: int) -> list[Record]:
    """Generate *n* synthetic web-form submissions (shape #3: flat, epoch time).

    Args:
        n: Number of submissions to create.

    Returns:
        A list of raw web-form records; ``satisfaction`` is occasionally ``None``.
    """
    feed: list[Record] = []
    for i in range(n):
        tone = random.choices(["negative", "neutral", "positive"], weights=[5, 3, 2])[0]
        issue = random.choice(ISSUE_TYPES)
        dt = BASE + timedelta(minutes=random.randint(0, 40000))
        sat = {"negative": 1, "neutral": 3, "positive": 5}[tone]
        rec: Record = {
            "id": 40000 + i,
            "name": fake.name(),
            "email": fake.email(),
            "issue_type": issue,
            "message": compose(tone, issue),
            "satisfaction": sat if random.random() > 0.15 else None,  # sometimes blank
            "submitted_at": int(dt.timestamp()),   # Unix epoch — a third time format!
        }
        feed.append(rec)
    return feed

webform_feed = make_webform_feed(N_WEBFORM)
print(f"{len(webform_feed)} web-form submissions")
print(json.dumps(webform_feed[0], indent=2))

Look closely at the three records above. They all represent **a customer getting in touch about a problem**, yet:

- the customer identifier is `customer_ref`, then `account`, then `name` + `email`
- the message is a *list of turns*, then a `body` field, then a `message` field
- the timestamp is **ISO 8601**, then a **UK date string**, then a **Unix epoch integer**

This is precisely why a rigid relational schema struggles — and precisely what the **Transform** stage exists to resolve.

In [ ]:
def cache_first(name: str, builder: Callable[[], list[Record]]) -> list[Record]:
    """Return cached raw data if present, otherwise build it and cache it.

    This 'cache-first' pattern makes the notebook reproducible and independent
    of the live source: the first run writes a JSON cache; later runs read it.

    Args:
        name: Cache key, used as the JSON filename.
        builder: Zero-argument function that produces the data on a cache miss.

    Returns:
        The list of raw records, whether loaded from cache or freshly built.
    """
    path = os.path.join(CACHE_DIR, f"{name}.json")
    if os.path.exists(path):
        with open(path) as f:
            print(f"Loaded '{name}' from cache.")
            return json.load(f)
    data = builder()
    with open(path, "w") as f:
        json.dump(data, f)
    print(f"Built '{name}' and wrote cache.")
    return data

raw = {
    "chat":    cache_first("chat",    lambda: chat_feed),
    "email":   cache_first("email",   lambda: email_feed),
    "webform": cache_first("webform", lambda: webform_feed),
}
print("Total raw records:", sum(len(v) for v in raw.values()))

> 📌 **Assessment connection**
>
> **Section B** of the summative asks for *anonymised screenshots of data at each stage of the pipeline*. The raw records above are your "before" screenshot. Take one here, one after Transform, and one of the loaded documents in Step 3 — and remember to **anonymise** any names, emails or account references (a common reason for lost marks). Our synthetic data is already safe, but real Pinnacle data would not be.

## 🏫 Step 2 — Transform

The Transform stage does the real work. We'll break it into three moves:

1. **Normalise** — map each channel's shape onto one *canonical* document.
2. **Enrich** — derive useful new fields (length, sentiment, priority).
3. **Validate** — quality-gate the records: reject the broken ones, de-duplicate the rest.

In [ ]:
def parse_when(value: int | str) -> datetime:
    """Parse the three timestamp formats the feeds use into a ``datetime``.

    Handles a Unix epoch (int), a UK ``dd/mm/YYYY HH:MM`` string, and an
    ISO 8601 string.

    Args:
        value: The raw timestamp, as an epoch integer or a string.

    Returns:
        The parsed ``datetime``.
    """
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value)
    for fmt in ("%d/%m/%Y %H:%M",):
        try:
            return datetime.strptime(value, fmt)
        except (ValueError, TypeError):
            pass
    return datetime.fromisoformat(value)   # ISO fallback

def normalise_chat(r: Record) -> Document:
    """Map a raw chat record onto the canonical document shape."""
    customer_turn = next((t["text"] for t in r["transcript"] if t["role"] == "customer"), "")
    return {
        "interaction_id": f"chat-{r['session_id']}",
        "channel": "chat",
        "received_at": parse_when(r["started"]),
        "customer_ref": r.get("customer_ref"),
        "message": customer_turn,
        "issue_type": None,                       # chat doesn't tag an issue type
        "source": {"session_id": r["session_id"], "turns": len(r["transcript"])},
    }

def normalise_email(r: Record) -> Document:
    """Map a raw email record onto the canonical document shape."""
    return {
        "interaction_id": f"email-{r['message_id']}",
        "channel": "email",
        "received_at": parse_when(r["date"]),
        "customer_ref": r.get("account"),
        "message": r.get("body", ""),
        "issue_type": r.get("subject", "").replace("Re: ", "").replace(" ", "_") or None,
        "source": {"message_id": r["message_id"], "from": r.get("from")},
    }

def normalise_webform(r: Record) -> Document:
    """Map a raw web-form record onto the canonical document shape."""
    return {
        "interaction_id": f"webform-{r['id']}",
        "channel": "webform",
        "received_at": parse_when(r["submitted_at"]),
        "customer_ref": None,                     # form has no account ref
        "message": r.get("message", ""),
        "issue_type": r.get("issue_type"),
        "source": {"id": r["id"], "satisfaction": r.get("satisfaction")},
    }

# Registry mapping each channel to its adapter. Typing the dict documents the
# contract every adapter must satisfy: Record in, Document out.
ADAPTERS: dict[str, Callable[[Record], Document]] = {
    "chat": normalise_chat, "email": normalise_email, "webform": normalise_webform,
}

# quick look at one normalised document
sample = normalise_email(raw["email"][0])
sample

In [ ]:
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_vader = SentimentIntensityAnalyzer()
PRIORITY_ISSUES = {"outage", "cancellation"}
PRIORITY_WORDS = {"down", "cancel", "leave", "furious", "unacceptable"}

def label_of(score: float) -> str:
    """Convert a VADER compound score into a coarse sentiment label.

    Args:
        score: VADER compound score in the range -1.0 to 1.0.

    Returns:
        "negative", "neutral" or "positive".
    """
    if score <= -0.3: return "negative"
    if score >=  0.3: return "positive"
    return "neutral"

def enrich(doc: Document) -> Document:
    """Add derived fields to a normalised document.

    Derives message length, a VADER sentiment score and label, and a coarse
    priority flag (high when a negative message concerns a priority issue or
    contains an urgent keyword).

    Args:
        doc: A normalised document (the output of a ``normalise_*`` adapter).

    Returns:
        A new document with ``message_length``, ``sentiment_score``,
        ``sentiment_label`` and ``priority`` added.
    """
    msg = (doc.get("message") or "").strip()
    score = _vader.polarity_scores(msg)["compound"]
    label = label_of(score)
    urgent = (label == "negative") and (
        doc.get("issue_type") in PRIORITY_ISSUES
        or any(w in msg.lower() for w in PRIORITY_WORDS)
    )
    doc = {**doc, "message": msg}
    doc["message_length"] = len(msg)
    doc["sentiment_score"] = round(score, 4)
    doc["sentiment_label"] = label
    doc["priority"] = "high" if urgent else "normal"
    return doc

enrich(sample)

In [ ]:
REQUIRED = ("interaction_id", "channel", "received_at")

def quality_gate(docs: list[Document]) -> tuple[list[Document], list[tuple[str | None, str]]]:
    """Split documents into those that pass basic quality checks and those that don't.

    Rejects records missing a required field or with an empty message, and
    removes duplicates by ``interaction_id``. This is where several of the six
    dimensions of data quality (completeness, uniqueness, validity) are enforced.

    Args:
        docs: Enriched documents to check.

    Returns:
        A ``(clean, rejected)`` pair, where ``rejected`` holds
        ``(interaction_id, reason)`` tuples explaining each removal.
    """
    clean: list[Document] = []
    rejected: list[tuple[str | None, str]] = []
    seen: set[str] = set()
    for d in docs:
        if any(not d.get(k) for k in REQUIRED):
            rejected.append((d.get("interaction_id"), "missing required field")); continue
        if not d.get("message"):
            rejected.append((d.get("interaction_id"), "empty message")); continue
        if d["interaction_id"] in seen:
            rejected.append((d["interaction_id"], "duplicate")); continue
        seen.add(d["interaction_id"])
        clean.append(d)
    return clean, rejected

# demonstrate on a deliberately broken batch
demo = [enrich(normalise_webform(raw["webform"][0])),
        {"interaction_id": "email-x", "channel": "email", "received_at": datetime.now(), "message": ""}]
clean_demo, rejected_demo = quality_gate(demo)
print("kept:", len(clean_demo), "| rejected:", rejected_demo)

The quality gate is where the module's **data-governance** thread appears. In the Level 4 *Data Engineering* module you met the **six dimensions of data quality** — completeness, uniqueness, timeliness, validity, accuracy, consistency. Our gate touches several of them directly: *completeness* (required fields present), *uniqueness* (de-duplication), *validity* (a non-empty message). A pipeline that silently loads broken records is worse than one that loudly rejects them.

In [ ]:
def transform(raw_sources: dict[str, list[Record]]) -> list[Document]:
    """Run the full Transform stage: normalise every channel, enrich, then quality-gate.

    Args:
        raw_sources: Mapping of channel name to its list of raw records.

    Returns:
        The list of clean, enriched documents ready to load.
    """
    normalised: list[Document] = []
    for channel, records in raw_sources.items():
        adapter = ADAPTERS[channel]
        normalised += [enrich(adapter(r)) for r in records]
    clean, rejected = quality_gate(normalised)
    print(f"Transformed {len(normalised)} records → {len(clean)} clean, {len(rejected)} rejected")
    return clean

clean_docs = transform(raw)
clean_docs[0]

In [ ]:
import pandas as pd
preview = pd.DataFrame(clean_docs)[
    ["interaction_id", "channel", "received_at", "issue_type", "sentiment_label", "priority"]
]
preview.head(8)

## 🏫 Step 3 — Load (into MongoDB)

Now we load the clean documents into MongoDB. Two design choices matter here.

**A reproducible backend.** A notebook shouldn't assume a database server is installed. We use **`mongomock`** — an in-memory implementation of the MongoDB API — so the code runs *identically* on your laptop, a hosted kernel, or a marker's machine, with nothing to install. It speaks the same `pymongo` API, so the code you learn is the code you'd write against a real deployment. Connecting to a real MongoDB or Atlas cluster is covered in the independent-study section.

**An idempotent load.** Running the pipeline twice must not create duplicate documents. We achieve this with an **upsert** keyed on the stable `interaction_id`: insert if new, update if seen. This is the single most important habit for a reproducible pipeline.

In [ ]:
from pymongo import MongoClient
from pymongo.collection import Collection

def get_client() -> MongoClient:
    """Return a MongoDB client — a real one if configured, else an in-memory mock.

    If ``MONGODB_URI`` is set and reachable, returns a real ``pymongo`` client;
    otherwise returns a ``mongomock`` client that implements the same API, so
    the rest of the notebook behaves identically either way. This is the same
    graceful-degradation pattern used elsewhere in the module.

    Returns:
        A client whose interface matches ``pymongo.MongoClient``. The mongomock
        fallback is API-compatible, though not a subclass, which is why the
        annotation names the interface rather than the concrete class.
    """
    uri = os.environ.get("MONGODB_URI")
    if uri:
        try:
            client = MongoClient(uri, serverSelectionTimeoutMS=2000)
            client.admin.command("ping")
            print("Connected to a real MongoDB deployment.")
            return client
        except Exception as e:
            print(f"Real MongoDB unreachable ({e!r}); falling back to mongomock.")
    import mongomock
    print("Using mongomock (in-memory) — reproducible, no server required.")
    return mongomock.MongoClient()

client = get_client()

In [ ]:
def load(docs: list[Document], client: MongoClient) -> Collection:
    """Idempotently load documents into MongoDB and ensure useful indexes exist.

    Uses an upsert keyed on ``interaction_id`` so re-running the pipeline never
    duplicates documents — the key habit behind a reproducible pipeline.

    Args:
        docs: Clean documents to store.
        client: A MongoDB client (real or mongomock).

    Returns:
        The ``interactions`` collection the documents were written to.
    """
    col = client["pinnacle"]["interactions"]
    for d in docs:
        col.update_one({"interaction_id": d["interaction_id"]}, {"$set": d}, upsert=True)
    for field in ("channel", "received_at", "issue_type", "sentiment_label"):
        col.create_index(field)
    return col

col = load(clean_docs, client)
print("Documents in collection:", col.count_documents({}))
print("Indexes:", sorted(col.index_information().keys()))

In [ ]:
# One stored document — note the flexible, self-describing shape (incl. nested `source`)
col.find_one({}, {"_id": 0})

### Why idempotency matters

An **idempotent** operation gives the same result whether you run it once or ten times. Our load upserts on `interaction_id`, so re-running the pipeline **updates** existing documents rather than duplicating them. Let's prove it.

In [ ]:
before = col.count_documents({})
load(clean_docs, client)          # run the load a second time
after = col.count_documents({})
print(f"Count before re-run: {before}  |  after re-run: {after}")
assert before == after, "Load is not idempotent!"
print("Idempotent ✔  — re-running the pipeline is safe.")

> 📌 **Assessment connection**
>
> Idempotency, a fixed random seed, and cache-first extraction are exactly the *"strategies for reproducibility"* named in the **LO5** rubric. When you write your technical report, don't just say your code is reproducible — **point to the specific mechanisms** (seed, cache, upsert) that make it so.

## 🏫 Step 4 — Query and aggregate for insight

The point of loading data well is to ask questions of it easily. MongoDB's **aggregation pipeline** is a sequence of stages — `$match`, `$group`, `$sort`, and so on — each feeding the next. It's the document-database equivalent of `GROUP BY`, and it reads like a pipeline because it *is* one.

In [ ]:
# Interaction volume and average sentiment, per channel
by_channel = list(col.aggregate([
    {"$group": {
        "_id": "$channel",
        "interactions": {"$sum": 1},
        "avg_sentiment": {"$avg": "$sentiment_score"},
    }},
    {"$sort": {"interactions": -1}},
]))
pd.DataFrame(by_channel).rename(columns={"_id": "channel"}).round(3)

In [ ]:
# Where are the high-priority contacts concentrated?
high_priority = list(col.aggregate([
    {"$match": {"priority": "high"}},
    {"$group": {"_id": "$issue_type", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
]))
pd.DataFrame(high_priority).rename(columns={"_id": "issue_type"})

### Top complaint terms

You might reach for MongoDB's server-side **`$text`** search here. It's a genuine feature of a real deployment — but our reproducible `mongomock` backend doesn't implement it (`NotImplementedError`). That's a useful reminder: **know the limits of the backend you're actually running on.** Where a feature isn't available, compute it another way. Here we count terms in Python over the negative messages; the independent-study section shows the `$text` approach for a real cluster.

In [ ]:
from collections import Counter
STOP = {"the", "and", "for", "with", "have", "this", "that", "been", "your", "pinnacle", "really"}

terms: Counter = Counter()
for d in col.find({"sentiment_label": "negative"}, {"message": 1}):
    for w in re.findall(r"[a-z]+", d["message"].lower()):
        if len(w) > 3 and w not in STOP:
            terms[w] += 1

pd.DataFrame(terms.most_common(10), columns=["term", "count"])

### Visualising the result

A well-labelled chart communicates far more quickly than a table. Here's average sentiment by channel — a natural thing for Pinnacle's operations team to watch.

In [ ]:
import matplotlib.pyplot as plt

NAVY, STEEL, AMBER = "#1B3A5C", "#2E6DA4", "#D4820A"
df_ch = pd.DataFrame(by_channel).rename(columns={"_id": "channel"}).sort_values("channel")

fig, ax = plt.subplots(figsize=(8, 4.5))
colours = [AMBER if v < 0 else STEEL for v in df_ch["avg_sentiment"]]
ax.bar(df_ch["channel"], df_ch["avg_sentiment"], color=colours, edgecolor=NAVY)
ax.axhline(0, color=NAVY, linewidth=0.8)
ax.set_title("Average customer sentiment by channel — Pinnacle Broadband", color=NAVY)
ax.set_ylabel("VADER compound score")
ax.set_xlabel("Channel")
for i, v in enumerate(df_ch["avg_sentiment"]):
    ax.text(i, v + (0.02 if v >= 0 else -0.04), f"{v:.2f}", ha="center", color=NAVY)
plt.tight_layout()
plt.show()

## 🏫 Putting it together — the pipeline as one function

Scattered cells are good for *learning* a pipeline; a **single, testable function** is what you *ship*. Wrapping Extract, Transform and Load into named functions with clear boundaries is what turns a notebook into robust, reproducible code — and it's directly what the LO5 rubric means by *"code presented as a product of design decisions rather than a transcript of trial and error."*

Notice, too, that every function in this notebook carries **type hints** and a **docstring**. The hints state precisely what goes in and what comes out (our `Record` and `Document` aliases make that contract readable at a glance); the docstring says *why* the function exists. Together they make the code self-documenting — another concrete, markable way to meet the LO5 requirement for *well-documented* code, and a habit professional data scientists are expected to have.

In [ ]:
def extract(use_cache: bool = True) -> dict[str, list[Record]]:
    """Extract stage: gather the three raw feeds, from cache by default.

    Args:
        use_cache: When True, use the cache-first helper; when False, rebuild
            every feed from scratch.

    Returns:
        Mapping of channel name to its list of raw records.
    """
    if not use_cache:
        return {"chat": make_chat_feed(N_CHAT),
                "email": make_email_feed(N_EMAIL),
                "webform": make_webform_feed(N_WEBFORM)}
    return {"chat": cache_first("chat", lambda: make_chat_feed(N_CHAT)),
            "email": cache_first("email", lambda: make_email_feed(N_EMAIL)),
            "webform": cache_first("webform", lambda: make_webform_feed(N_WEBFORM))}

def run_pipeline(client: MongoClient) -> Collection:
    """Run the end-to-end ETL pipeline: extract → transform → load.

    Args:
        client: A MongoDB client to load the documents into.

    Returns:
        The populated ``interactions`` collection.
    """
    raw_sources = extract()
    docs = transform(raw_sources)
    col = load(docs, client)
    print(f"Pipeline complete — {col.count_documents({})} documents in the store.")
    return col

col = run_pipeline(get_client())

## 🏫 Robustness — testing the pipeline

Reproducible, robust code is **tested** code. We use `ipytest` to run unit tests inside the notebook. Good tests here check the parts most likely to break silently: the enrichment logic, the idempotent load, and the quality gate.

In [ ]:
import ipytest
ipytest.autoconfig()

In [ ]:
%%ipytest -qq

def test_enrich_labels_clear_negative() -> None:
    """A clearly angry message is labelled negative and flagged high priority."""
    doc = enrich({"interaction_id": "t1", "channel": "chat",
                  "received_at": datetime.now(),
                  "message": "Absolutely appalling, I am furious and want to cancel."})
    assert doc["sentiment_label"] == "negative"
    assert doc["priority"] == "high"          # negative + a priority word

def test_load_is_idempotent() -> None:
    """Loading the same document twice results in a single stored copy."""
    import mongomock
    c = mongomock.MongoClient()
    d = [{"interaction_id": "only-one", "channel": "email",
          "received_at": datetime.now(), "message": "hello"}]
    load(d, c); load(d, c)                     # twice
    assert c["pinnacle"]["interactions"].count_documents({}) == 1

def test_quality_gate_rejects_empty_message() -> None:
    """A document with an empty message is rejected by the quality gate."""
    bad = [{"interaction_id": "b1", "channel": "chat",
            "received_at": datetime.now(), "message": ""}]
    clean, rejected = quality_gate(bad)
    assert clean == [] and len(rejected) == 1

> 🔍 **Design decision — when NOT to reach for NoSQL**
>
> It would be a poor Level 6 answer to present document databases as universally superior. They are the right tool *here* because the data is varied, nested and evolving. They are the **wrong** tool when:
>
> - You need **multi-record transactions** with strong guarantees (a document store's flexibility comes at the cost of the relational world's mature ACID transactions across tables).
> - Your data is **uniform and highly relational** — customers, orders, line-items with strict foreign keys — where joins and a fixed schema are a *feature*, catching errors early.
> - You need **ad-hoc reporting** in SQL for analysts who don't write aggregation pipelines.
>
> The schema flexibility that helps us ingest three messy feeds is the *same* property that lets bad data slip in unnoticed — which is exactly why we built a quality gate. **Every design choice buys something and costs something; naming both is what the report is marking.**

## 📖 Independent study

The material below extends the session. You're not expected to cover it in class, but working through it will strengthen your summative answer.

**1. Connect to a real MongoDB / Atlas cluster.** Everything you've learned uses the `pymongo` API, so moving to a real deployment is a one-line change: set a `MONGODB_URI` environment variable (e.g. a free Atlas M0 connection string) and `get_client()` will use it automatically. *Never hard-code credentials in a notebook you might submit* — that's both a security failure and an anonymisation failure.

**2. Server-side text search.** On a real deployment, a text index lets you query with `$text`:
```python
col.create_index([("message", "text")])
list(col.find({"$text": {"$search": "outage refund"}}))
```
Compare the results with our Python term-frequency approach. When would server-side search be worth the index cost?

**3. The other NoSQL families.** Sketch how you'd store the *same* Pinnacle data in a key–value store (Redis) and a graph database (Neo4j). What questions become easy, and what becomes hard?

**4. The six dimensions of data quality.** Extend `quality_gate` to measure *timeliness* (reject records older than a threshold) and *validity* (check `received_at` is a real date not in the future). Report a quality score per channel.

In [ ]:
# Independent study — this cell is safe to run: it no-ops unless MONGODB_URI is set.
uri = os.environ.get("MONGODB_URI")
if not uri:
    print("Set MONGODB_URI (e.g. an Atlas connection string) to run this section "
          "against a real deployment. The pipeline code above needs no other change.")
else:
    real = MongoClient(uri)
    real_col = load(clean_docs, real)
    real_col.create_index([("message", "text")])
    print("Text search hits:",
          real_col.count_documents({"$text": {"$search": "outage"}}))

## 📝 Your turn — exercises

1. **Add a fourth channel.** Pinnacle launches a WhatsApp feed: `{ "wa_id", "phone", "text", "ts" (epoch) }`. Write `normalise_whatsapp`, register it in `ADAPTERS`, and confirm the pipeline ingests it **with no other changes**. What does that tell you about the document-store approach?

2. **Sharpen the priority rule.** Right now `priority` uses a small keyword set. Redesign it — perhaps weighting very negative sentiment more heavily, or flagging repeated contacts from the same `customer_ref`. Justify your rule in two sentences.

3. **Enrich the quality report.** Make `quality_gate` return a small summary (counts by rejection reason, per channel) and display it as a DataFrame. Which channel is dirtiest, and why might that be?

4. **ETL → ELT.** Re-shape the pipeline to load the *raw* records first, then run the transform as an aggregation/update inside the store. What did you gain and lose?

## Summary

We took three incompatible raw feeds and turned them into one clean, queryable MongoDB collection using a repeatable, tested ETL pipeline.

| Stage | What it did | Key tools |
|---|---|---|
| **Extract** | Gathered three heterogeneous feeds; cached them | `faker`, seed, cache-first |
| **Transform** | Normalised shapes, enriched, quality-gated | per-channel adapters, `vaderSentiment` |
| **Load** | Idempotent upsert into a document store; indexed | `pymongo` / `mongomock` |
| **Analyse** | Aggregation pipelines + a visualisation | `$group` / `$match`, `matplotlib` |

### Key takeaways

1. **Document stores fit unstructured data** because they don't force a single schema onto varied, nested, evolving records — but that same flexibility is why a **quality gate** is essential.
2. **ETL vs ELT is a design decision**, not a default. Say which you chose and why.
3. **Idempotency, seeds and caching** are the concrete mechanisms behind "reproducible" — name them.
4. **Wrap the pipeline in tested, type-hinted, documented functions.** Robust code reads as deliberate design, not trial and error.
5. **No tool is universally right.** The Level 6 skill is naming what your choice costs as clearly as what it buys.

## Assessment mapping

| Learning outcome | Where this notebook addresses it |
|---|---|
| **LO2 — process unstructured data** *(20%)* | Whole pipeline: ingesting and storing varied, semi-structured records |
| **LO4 — Python for analysis & visualisation** *(20%)* | Aggregation pipelines, term-frequency analysis, the sentiment chart |
| **LO5 — reproducible & robust code** *(10%)* | Seeds, cache-first extract, idempotent load, type hints, docstrings, `ipytest` tests |
| **LO1 — NLP / text analysis** *(20%)* | Light touch: VADER enrichment in Transform (developed fully in Topic 6) |
| **LO3 — web scraping** *(20%)* | Light touch: the raw feeds represent the output of Topics 1–4 collection |
| **Academic writing & referencing** *(10%)* | The 🔍 design-decision boxes model the reasoning the technical report rewards |

## What's next?

Pinnacle Broadband's support interactions are now sitting in a clean, indexed store — and we'll keep working with them. **Topic 6** takes this same collection into serious text analytics: TF-IDF, named-entity recognition and topic modelling, turning the messages you've just pipelined into structured insight.

The habits from this topic — a repeatable pipeline, reproducible code, and a clear-eyed appraisal of every design choice — carry through the rest of the module.